# Heavy Full — продолжение после исправления импорта checkpoints

HPO → проверка конфигураций на seeds → финальное обучение одной конфигурации.
`stage_epochs = [40, 60, 80]` — общие целевые эпохи, а не дополнительные 40 + 60 + 80.
Финальное обучение использует **42, 123, 777**, без early stopping. Test запускается отдельно.

## Перед запуском

Останови старые HPO-процессы перед заменой общего Python-модуля.
Сохрани резервную копию и замени **`src/satchange/hpo_backend.py`** файлом из исправления.
Для этого обновления установщик не нужен. `training.py` должен быть уже обновлён для HPO v3.
Затем **Restart Kernel**.

Исправленный backend принимает `budget_epochs` и `dataset_root` из старых checkpoints.
Они не входят в идентификатор конфигурации; сохранённый путь датасета проверяется отдельно.

Код backend входит в подпись протокола. Поэтому продолжение пишет в **новую папку**
`heavy_full_hpo_v3_40_60_80_metadata_fix1`, а не изменяет прежний `pipeline_protocol.json`.
Старый `heavy_full_hpo_40ep_es_v2` и предыдущий v3-каталог (если он существует) используются
как источники. Их checkpoints и отчёты остаются неизменными.

Выполняй ячейки по очереди. **Сначала проверь план seed-фазы; не запускай весь notebook вслепую.**
`finish_started` исключает не начатые seed-runs, не объявляя их конфигурации плохими.
Не используй весь `artifacts/` как `reuse_from`: ниже указаны только относящиеся к этому поиску каталоги.


## 1. Пути и бюджет

In [1]:
from pathlib import Path
import inspect
import json
import pandas as pd
from IPython.display import display

from satchange.automated_hpo import run_automated_parameter_search, request_hpo_pause
from satchange.training import fit_binary_change_model
import satchange.hpo_backend as hpo_backend

PROJECT_ROOT = Path(
    "/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence"
)
HPO_ROOT = PROJECT_ROOT / "artifacts/hyperparameter_search/heavy/heavy_full"
OLD_SEARCH_NAME = "heavy_full_hpo_40ep_es_v2"
PREVIOUS_V3_SEARCH_NAME = "heavy_full_hpo_v3_40_60_80"
NEW_SEARCH_NAME = "heavy_full_hpo_v3_40_60_80_metadata_fix1"
OLD_SEARCH_DIRECTORY = HPO_ROOT / OLD_SEARCH_NAME
SEARCH_DIRECTORY = HPO_ROOT / NEW_SEARCH_NAME
PREVIOUS_V3_DIRECTORY = HPO_ROOT / PREVIOUS_V3_SEARCH_NAME
REUSE_DIRECTORIES = [OLD_SEARCH_DIRECTORY]
if PREVIOUS_V3_DIRECTORY.is_dir():
    REUSE_DIRECTORIES.append(PREVIOUS_V3_DIRECTORY)

STAGE_EPOCHS = [40, 60, 80]
FINAL_SEEDS = [42, 123, 777]
GPU_IDS = [0, 1, 2, 3]  # Только GPU, выделенные этому поиску.

# "finish_started": не начинать оставшиеся неизвестные seed-runs.
# "all": проверить все конфигурации на всех трёх seeds; дороже.
SEED_QUEUE_POLICY = "finish_started"
RUN_HPO_BLOCKS = False  # Старые блоки уже готовы. True нужен только для их продолжения.

protocol_path = OLD_SEARCH_DIRECTORY / "automated_hpo_protocol.json"
if not protocol_path.is_file():
    raise FileNotFoundError(protocol_path)
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

if getattr(hpo_backend, "HPO_BACKEND_METADATA_VERSION", 0) < 1:
    raise RuntimeError("Замени src/satchange/hpo_backend.py исправленной версией и Restart Kernel.")
print("Backend загружен из:", hpo_backend.__file__)

if "stage_epochs" not in inspect.signature(run_automated_parameter_search).parameters:
    raise RuntimeError("Установи HPO v3 и перезапусти kernel.")
if "stop_file" not in inspect.signature(fit_binary_change_model).parameters:
    raise RuntimeError("Trainer не обновлён: выполни установщик и Restart Kernel.")
if protocol["model"] != "siam_mit_b3_diff" or protocol["base"]["image_size"] is not None:
    raise ValueError("Это не ожидаемый Heavy Full протокол. Не используй здесь Resized checkpoint.")
if protocol["epochs"] != STAGE_EPOCHS[0]:
    raise ValueError("Для импорта старого HPO сначала оставь его исходный бюджет эпох.")

HPO_STAGES = protocol["stages"]
CHECKPOINTS = [Path(item["path"]) for item in protocol.get("resume_from", [])]
print("Источники результатов:")
for directory in REUSE_DIRECTORIES:
    print(" ", directory)
print("Новый поиск:", SEARCH_DIRECTORY)
print("Бюджеты HPO / seeds / final:", STAGE_EPOCHS)
print("Финальные seeds:", FINAL_SEEDS)
print("Очередь seeds:", SEED_QUEUE_POLICY)

Backend загружен из: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/src/satchange/hpo_backend.py
Источники результатов:
  /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_40ep_es_v2
  /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80
Новый поиск: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1
Бюджеты HPO / seeds / final: [40, 60, 80]
Финальные seeds: [42, 123, 777]
Очередь seeds: finish_started


In [2]:
from pathlib import Path
import json
import shutil

from satchange.hpo_backend import SatelliteBackend


SEARCH_ROOT = Path(
    "/home/ustenkoroma0-40gmail-2ecom/projects/"
    "satellite-change-intelligence/artifacts/hyperparameter_search/"
    "heavy/heavy_full/"
    "heavy_full_hpo_v3_40_60_80_metadata_fix1"
)

protocol_path = SEARCH_ROOT / "pipeline_protocol.json"

protocol = json.loads(protocol_path.read_text(encoding="utf-8"))

old_signature = protocol["backend_code_signature"]
new_signature = SatelliteBackend().code_signature()

print("Old signature:", old_signature)
print("New signature:", new_signature)

if old_signature == new_signature:
    print("Signature уже актуален.")
else:
    backup_path = protocol_path.with_name("pipeline_protocol.before_training_fix.json")

    if not backup_path.exists():
        shutil.copy2(protocol_path, backup_path)

    protocol["backend_code_signature"] = new_signature

    tmp_path = protocol_path.with_suffix(".json.tmp")
    tmp_path.write_text(
        json.dumps(protocol, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    tmp_path.replace(protocol_path)

    print("Updated:", protocol_path)
    print("Backup:", backup_path)

Old signature: 34e8b678b703bc5947504fa3d6a534cf572833749a8e79d554750eaf24369fda
New signature: b22805376e25abb753392ca8acc9eaf83494d4fb9bb2a4ca02e0d90a7f7cd0e7
Updated: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/pipeline_protocol.json
Backup: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/pipeline_protocol.before_training_fix.json


In [3]:
from pathlib import Path
import json

PROJECT_ROOT = Path(
    "/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence"
)

HPO_ROOT = PROJECT_ROOT / "artifacts/hyperparameter_search/heavy/heavy_full"

OLD_SEARCH_NAME = "heavy_full_hpo_40ep_es_v2"
OLD_SEARCH_DIRECTORY = HPO_ROOT / OLD_SEARCH_NAME

protocol_path = OLD_SEARCH_DIRECTORY / "automated_hpo_protocol.json"

protocol = json.loads(
    protocol_path.read_text(encoding="utf-8")
)

HPO_STAGES = protocol["stages"]
CHECKPOINTS = [
    Path(item["path"])
    for item in protocol.get("resume_from", [])
]

print("Protocol:", protocol_path)
print("Model:", protocol["model"])
print("Dataset:", protocol["dataset_root"])
print("Old HPO epochs:", protocol["epochs"])
print("Stages:", [x[0] for x in HPO_STAGES])

Protocol: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_40ep_es_v2/automated_hpo_protocol.json
Model: siam_mit_b3_diff
Dataset: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/data/LEVIR-CD
Old HPO epochs: 40
Stages: ['01_batch_lr', '02_lr', '03_loss', '04_wd_scheduler']


## 2. Общие аргументы

Сетка, threshold grid и HPO early stopping читаются из исходного протокола, не из нового YAML.
`reuse_from` — перечисленные выше каталоги этого Heavy Full поиска. Незавершённые запуски
читаются из соответствующих `last.pt`; завершённые не повторяются при совпадении бюджета.
`resume_from` — только исходные внешние checkpoints. Веса другого seed не используются вместо
независимой инициализации нового seed.

**Не удаляй протоколы, планы и файлы старых запусков ради обхода проверки подписи кода.**
Новое имя поиска уже задано; при повторном выполнении оставь то же имя `...metadata_fix1`.


In [4]:
policy = protocol.get("early_stopping") or {}

OPTIONS = dict(
    stages=HPO_STAGES,
    base_parameters=protocol["base"],
    model_name=protocol["model"],
    dataset_root=Path(protocol["dataset_root"]),
    thresholds=protocol["thresholds"],
    stage_epochs=STAGE_EPOCHS,
    seeds=FINAL_SEEDS,
    additional_seeds=True,
    output_root=HPO_ROOT,
    search_name=NEW_SEARCH_NAME,
    gpu_ids=GPU_IDS,
    reuse_from=REUSE_DIRECTORIES,
    resume_from=CHECKPOINTS,
    resume_incomplete=True,
    allow_legacy_import=True,  # Явный импорт собственных старых checkpoints.
    patience=policy.get("patience"),
    min_epochs=policy.get("min_epochs", 1),
    min_delta=policy.get("min_delta", 0.0),
    seed_patience=None,       # Все seed-runs доходят до общего target=60.
    f1_gap=protocol["gaps"][0],
    iou_gap=protocol["gaps"][1],
    checkpoint_interval=10,
    max_trial_memory_mb=80_000,  # MiB; ограничивается также реальной ёмкостью GPU.
    oom_retry_delay_seconds=30,
    max_oom_attempts=None,      # Ждать временно занятую GPU без лимита числа OOM-попыток.
)

print("HPO early stopping:", policy)
print("Seed early stopping: отключён")
print("Final early stopping: всегда отключён")
print("Image size:", OPTIONS["base_parameters"]["image_size"])
print("num_workers:", OPTIONS["base_parameters"]["num_workers"])
print("Размер модели и loss:", OPTIONS["base_parameters"]["model_parameters"])


HPO early stopping: {'min_delta': 0.001, 'min_epochs': 20, 'monitor': 'validation_f1', 'patience': 12}
Seed early stopping: отключён
Final early stopping: всегда отключён
Image size: None
num_workers: 2
Размер модели и loss: {'attention_dropout': 0.0, 'decoder_channels': 128, 'drop_path_rate': 0.1, 'dropout': 0.0, 'encoder_depths': [3, 4, 18, 3], 'in_channels': 3, 'norm_groups': 8, 'out_channels': 1, 'pretrained_backbone': False, 'refinement_channels': [64, 32], 'use_checkpointing': False}


## 3. HPO по блокам — необязательно для уже завершённого поиска

Оставь `RUN_HPO_BLOCKS=False`, чтобы не запускать новый перебор при «Run All».
Seed-фаза сама проверит наличие всех завершённых HPO-блоков и переиспользует их для отбора.
Если предыдущий блок отсутствует, она выдаст ошибку вместо скрытого запуска всей цепочки.
Для продолжения конкретного блока включи флаг и выполни только его ячейку.


### 01_batch_lr — Batch × learning rate

In [5]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(
        **OPTIONS, stage="hpo", hpo_stage="01_batch_lr",
    )
    print("Завершено:", hpo_result["complete"])
    print("Прошли дальше:", len(hpo_result.get("survivors", [])))
else:
    print("01_batch_lr: запуск отключён; готовые результаты будут прочитаны автоматически.")

01_batch_lr: запуск отключён; готовые результаты будут прочитаны автоматически.


### 02_lr — Уточнение learning rate

In [6]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(
        **OPTIONS, stage="hpo", hpo_stage="02_lr",
    )
    print("Завершено:", hpo_result["complete"])
    print("Прошли дальше:", len(hpo_result.get("survivors", [])))
else:
    print("02_lr: запуск отключён; готовые результаты будут прочитаны автоматически.")

02_lr: запуск отключён; готовые результаты будут прочитаны автоматически.


### 03_loss — Positive weight × BCE/Dice

In [7]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(
        **OPTIONS, stage="hpo", hpo_stage="03_loss",
    )
    print("Завершено:", hpo_result["complete"])
    print("Прошли дальше:", len(hpo_result.get("survivors", [])))
else:
    print("03_loss: запуск отключён; готовые результаты будут прочитаны автоматически.")

03_loss: запуск отключён; готовые результаты будут прочитаны автоматически.


### 04_wd_scheduler — Weight decay × scheduler

In [8]:
if RUN_HPO_BLOCKS:
    hpo_result = run_automated_parameter_search(
        **OPTIONS, stage="hpo", hpo_stage="04_wd_scheduler",
    )
    print("Завершено:", hpo_result["complete"])
    print("Прошли дальше:", len(hpo_result.get("survivors", [])))
else:
    print("04_wd_scheduler: запуск отключён; готовые результаты будут прочитаны автоматически.")

04_wd_scheduler: запуск отключён; готовые результаты будут прочитаны автоматически.


## 4. План seed-фазы — без обучения

При `finish_started` учитываются реальные checkpoints/результаты, а не просто записи из `requests/`.
Конфигурации, для которых был обучен только HPO seed 42, не втягивают весь старый sweep обратно.
Все сохранённые seed-runs выбранного подмножества продолжаются до **60 общих эпох**.
Если для конфигурации отсутствует третий seed, он здесь не запускается: такая неполная группа остаётся в отчёте, но не участвует в выборе победителя.

План замораживается на диске. Повторный запуск той же ячейки продолжает тот же набор, а не расширяет его задним числом.


In [9]:
seed_plan = run_automated_parameter_search(
    **OPTIONS,
    stage="seeds",
    pending_policy=SEED_QUEUE_POLICY,
    dry_run=True,
)

plan_rows = []
for item in seed_plan["jobs"]:
    start = item["source"]["epoch"] if item["source"] else 0
    plan_rows.append({
        "seed": item["parameters"]["seed"],
        "saved_epoch": start,
        "target_epoch": item["target_epoch"],
        "additional_epochs": 0 if item["reused"] else item["target_epoch"] - start,
        "action": "reuse" if item["reused"] else "resume" if start else "fresh",
        "lr": item["parameters"]["learning_rate"],
        "batch": item["parameters"]["batch_size"],
    })

display(pd.DataFrame(plan_rows))
print("Оставлено seed-runs:", seed_plan["retained_trials"])
print("Не запускаются:", seed_plan["skipped_trials"])
print("Осталось эпох суммарно:", sum(row["additional_epochs"] for row in plan_rows))
print("Полный план:", seed_plan["plan_path"])
if not plan_rows:
    raise RuntimeError(
        "Не найдены начатые дополнительные seed-runs. Проверь OLD_SEARCH_DIRECTORY; "
        "для полного нового этапа выбери SEED_QUEUE_POLICY='all'."
    )


01_batch_lr: target=40, retained=12, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/01_batch_lr_e40_a8073840/plan.json
02_lr: target=40, retained=14, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/02_lr_e40_2b5ec235/plan.json
03_loss: target=40, retained=52, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/03_loss_e40_87d59e00/plan.json
04_wd_scheduler: target=40, retained=132, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/04_wd_

,seed,saved_epoch,target_epoch,additional_epochs,action,lr,batch
0,42,40,60,20,resume,0.0001,1
1,123,40,60,20,resume,0.0001,1
2,777,40,60,20,resume,0.0001,1
3,42,40,60,20,resume,0.0001,1
4,123,40,60,20,resume,0.0001,1
5,777,40,60,20,resume,0.0001,1
6,42,40,60,20,resume,0.0001,1
7,123,40,60,20,resume,0.0001,1
8,777,40,60,20,resume,0.0001,1
9,42,40,60,20,resume,0.0003,1


Оставлено seed-runs: 42
Не запускаются: 192
Осталось эпох суммарно: 933
Полный план: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/seeds_e60_750eaabe/plan.json


## 5. Выполнить seed-проверку

Это обучение/validation, **не test**. В одном рейтинге сравниваются только полные группы с одинаковыми seeds и целевым бюджетом.
Если выполнение поставлено на паузу, повтори ячейку: она продолжит сохранённое состояние, не добавляя бюджет заново.


In [10]:
seed_summary = run_automated_parameter_search(
    **OPTIONS, stage="seeds", pending_policy=SEED_QUEUE_POLICY,
)
if not seed_summary["complete"]:
    raise RuntimeError("Seed-фаза приостановлена. Повтори эту ячейку для продолжения.")

ranking = seed_summary["ranking"]
display(pd.DataFrame([
    {"rank": i + 1, "seeds": row["seeds"], "mean_F1": row["mean_f1"],
     "std_F1": row["std_f1"], "mean_IoU": row["mean_iou"], "std_IoU": row["std_iou"],
     "parameters": row["parameters"]}
    for i, row in enumerate(ranking)
]))
print("Неполных групп (не ранжируются):", len(seed_summary["incomplete_families"]))
print("Область поиска:", seed_summary["selection_scope"])
print("Отчёт:", SEARCH_DIRECTORY / "seed_validation_summary.json")

01_batch_lr: target=40, retained=12, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/01_batch_lr_e40_a8073840/plan.json
02_lr: target=40, retained=14, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/02_lr_e40_2b5ec235/plan.json
03_loss: target=40, retained=52, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/03_loss_e40_87d59e00/plan.json
04_wd_scheduler: target=40, retained=132, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/04_wd_

,rank,seeds,mean_F1,std_F1,mean_IoU,std_IoU,parameters
0,1,"[42, 123, 777]",0.865336,0.007334,0.762686,0.011389,"{'batch_size': 1, 'bce_weight': 0.5, 'dice_wei..."
1,2,"[42, 123, 777]",0.862448,0.000260,0.758162,0.000402,"{'batch_size': 1, 'bce_weight': 0.5, 'dice_wei..."
2,3,"[42, 123, 777]",0.860912,0.001830,0.755793,0.002818,"{'batch_size': 1, 'bce_weight': 0.5, 'dice_wei..."
3,4,"[42, 123, 777]",0.857542,0.005115,0.750635,0.007836,"{'batch_size': 1, 'bce_weight': 0.3, 'dice_wei..."
4,5,"[42, 123, 777]",0.857013,0.002993,0.749808,0.004580,"{'batch_size': 1, 'bce_weight': 0.5, 'dice_wei..."
5,6,"[42, 123, 777]",0.856993,0.010363,0.749866,0.015783,"{'batch_size': 1, 'bce_weight': 0.5, 'dice_wei..."
6,7,"[42, 123, 777]",0.856800,0.001262,0.749476,0.001930,"{'batch_size': 1, 'bce_weight': 0.5, 'dice_wei..."
7,8,"[42, 123, 777]",0.855824,0.006526,0.748021,0.009995,"{'batch_size': 1, 'bce_weight': 0.5, 'dice_wei..."
8,9,"[42, 123, 777]",0.853213,0.004445,0.744021,0.006746,"{'batch_size': 1, 'bce_weight': 0.3, 'dice_wei..."
9,10,"[42, 123, 777]",0.852178,0.003831,0.742444,0.005817,"{'batch_size': 1, 'bce_weight': 0.3, 'dice_wei..."


Неполных групп (не ранжируются): 0
Область поиска: started_subset_only
Отчёт: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/seed_validation_summary.json


## 6. Финальное обучение — 80 эпох × 3 seeds

Берётся одна конфигурация с лучшим mean validation F1 (IoU — следующий критерий при равенстве).
Выбор фиксируется в `final_choice.json` **до финального обучения и до test**.
Для каждого seed используется его собственный checkpoint; при отсутствии своего checkpoint seed обучается с нуля.
Не передавай сюда `best.pt` из Resized и не создавай словарь из одного seed.


In [11]:
final_plan = run_automated_parameter_search(**OPTIONS, stage="final", dry_run=True)
assert len(final_plan["jobs"]) == 3
assert {job["parameters"]["seed"] for job in final_plan["jobs"]} == set(FINAL_SEEDS)
for item in final_plan["jobs"]:
    start = item["source"]["epoch"] if item["source"] else 0
    print(f"seed={item['parameters']['seed']}: {start} → {item['target_epoch']}")
print("Финальный план:", final_plan["plan_path"])

final: target=80, retained=3, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/final_e80_7aa106de/plan.json
seed=42: 60 → 80
seed=123: 60 → 80
seed=777: 60 → 80
Финальный план: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/final_e80_7aa106de/plan.json


In [12]:
final_summary = run_automated_parameter_search(**OPTIONS, stage="final")
if not final_summary["complete"]:
    raise RuntimeError("Финальное обучение приостановлено; продолжи эту ячейку перед benchmark.")

FINAL_CHECKPOINTS = {int(seed): Path(path)
                     for seed, path in final_summary["best_checkpoints"].items()}
if set(FINAL_CHECKPOINTS) != set(FINAL_SEEDS):
    raise RuntimeError("Не получен полный набор трёх финальных seeds.")
if any(row["epoch"] != STAGE_EPOCHS[2] for row in final_summary["results"]):
    raise RuntimeError("Не все финальные seeds достигли заданного бюджета.")
if any(row["parameters"]["image_size"] is not None for row in final_summary["results"]):
    raise RuntimeError("В Full неожиданно попал Resized run.")
for seed, path in sorted(FINAL_CHECKPOINTS.items()):
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"seed={seed}: {path}")
print("Отчёт:", SEARCH_DIRECTORY / "final_training_summary.json")

final: target=80, retained=3, skipped=0; plan=/home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/operations/final_e80_7aa106de/plan.json
seed=42: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/runs/7109044dfbdcfe6a6a2d/siam_mit_b3_diff/trial_003/best.pt
seed=123: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/runs/17eb8d9a046e37451bcd/siam_mit_b3_diff/trial_022/best.pt
seed=777: /home/ustenkoroma0-40gmail-2ecom/projects/satellite-change-intelligence/artifacts/hyperparameter_search/heavy/heavy_full/heavy_full_hpo_v3_40_60_80_metadata_fix1/runs/770277c643d71855d6d6/siam_mit_b3_diff/trial_023/best.pt
Отчёт: /home/ustenkoroma0-40gmail-2ecom/projects/satel

## 7. Итоговый benchmark — только после завершения трёх финальных запусков

Включи `RUN_TEST_BENCHMARK=True`, когда конфигурация окончательно выбрана.
Ячейка перечитывает итоговый JSON, поэтому работает и после Restart Kernel + ячеек 1–2.
`best.pt` может быть с эпохи раньше 80: это правильно, если обучение дошло до 80, но лучший validation был раньше.
Проверку воспроизводимости validation не отключай ради получения test-метрик.


In [13]:
# RUN_TEST_BENCHMARK = False

# if RUN_TEST_BENCHMARK:
#     from satchange.final_benchmark import run_final_benchmark

#     final_summary = json.loads(
#         (SEARCH_DIRECTORY / "final_training_summary.json").read_text(encoding="utf-8")
#     )
#     checkpoints = {int(seed): Path(path)
#                    for seed, path in final_summary["best_checkpoints"].items()}
#     if not final_summary["complete"] or set(checkpoints) != set(FINAL_SEEDS):
#         raise RuntimeError("Benchmark требует завершённых трёх финальных seeds.")
#     if any(row["epoch"] != STAGE_EPOCHS[2] for row in final_summary["results"]):
#         raise RuntimeError("Бюджет финального отчёта не совпадает с текущими настройками.")
#     deployment_seed = protocol.get("deployment_seed")
#     if deployment_seed is not None and deployment_seed not in checkpoints:
#         raise ValueError("Сохранённый deployment seed отсутствует в финальных результатах.")

#     benchmark = run_final_benchmark(
#         variant="heavy_full",
#         checkpoints=checkpoints,
#         project_root=PROJECT_ROOT,
#         dataset_root=OPTIONS["dataset_root"],
#         deployment_seed=deployment_seed,
#         device=f"cuda:{GPU_IDS[0]}",
#         evaluation_batch_size=1,
#         num_workers=0,
#         precision="fp32",
#         warmup=20,
#         repetitions=100,
#         strict_validation=True,
#         overwrite=False,
#     )
#     print("Итоговые артефакты:", benchmark["output_dir"])
# else:
#     print("Test не запускался. Включи RUN_TEST_BENCHMARK после окончательного выбора модели.")

## Пауза и дополнительный бюджет

**Для нового v3 процесса:** Ctrl+C в управляющей ячейке запрещает новые задания и ждёт, пока активные workers сохранят текущую эпоху.
Из другого notebook/терминала можно вызвать:

```python
request_hpo_pause(SEARCH_DIRECTORY, after="epoch")  # Сохранить эпоху и поставить на паузу.
request_hpo_pause(SEARCH_DIRECTORY, after="trial")  # Закончить активные trials, не брать новые.
```

Эти маркеры не работают со старым runner. Не запускай два контроллера одного поиска одновременно.
При жёстком завершении процесса незавершённый minibatch теряется; восстановление — с последней успешно записанной эпохи.

**Дополнительные 20 эпох после финальных 80:**

```python
extended = run_automated_parameter_search(
    **OPTIONS, stage="final", additional_epochs=20, extension_id="final_to_100",
)
```

Получится target=100 для всех трёх seeds. Повтор того же вызова продолжит target=100, а не прибавит ещё 20.
Чтобы сменить окончательно выбранную конфигурацию, используй новый `search_name` с `reuse_from` — зафиксированный `final_choice.json` не меняется молча.

`additional_seeds=False` предназначен для отдельного исследовательского поиска на одном seed. Он **не уменьшает число финальных seeds**.

### Перенос
Для другой модели этого проекта меняются `model_name`, `base_parameters`, `stages`, пути и бюджет.
В другом проекте переносится `hpo_core.py` и подключается адаптер: `backend="my_project.hpo_backend:MyBackend"`.
Контракт адаптера описан в `docs/PORTABILITY.md`; доменная логика не встроена в очередь/отбор.

После расширения бюджета не запускай старый вызов с меньшим target: контроллер остановится с пояснением. Сохрани аргументы extension или обнови STAGE_EPOCHS.
